# Time-Resolved RSA
NRR_RW011 / target_B / NPRW / cond 5,7,10,11 + ctrl BR002

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [2]:
# Paths 
DATA_ROOT = Path(r'V:\Current Project Databases - NHP\2025 Cerebellum prosthesis\Nike')
SESSION = '20251203_NRR_RW011'
PERI = DATA_ROOT / SESSION / 'results' / 'checkpoints' / 'PeriStim'

stim_dir = PERI / 'stim_reaches' / 'target_B'
ctrl_dir = PERI / 'control_reaches' / 'target_B'

print('stim NPZs:', len(list(stim_dir.glob('*.npz'))))
print('ctrl NPZs:', len(list(ctrl_dir.glob('*.npz'))))

stim NPZs: 20
ctrl NPZs: 3


In [ ]:
# Condition definitions. Electrode ranges from Bryan's cond_label_extras
# (bryan_scripts/RSA_consistency/RSA_poststim_grouped_up.ipynb).
# Everything else (freq, current, port, at-rest flag) is read off each NPZ's own metadata.

# Which block is the reaching 400 Hz / ch33-64 cell.
#   BR7 -> Bryan labels it "400Hz (33-64)", NPZ has is_at_rest=False, hand span 3.28
#   BR9 -> Bryan labels it "32 ch (Rest, 33-64)", is_at_rest=True, hand span 0.34, in at_rest/
# Both independent sources point at 7 for the reaching 2x2. Flip this to test the other.
GROUP_HI_BR = 7

COND_MAP = {
    10:          (130, '1-32',  'c10: 130Hz ch1-32'),
    5:           (400, '1-32',  'c5: 400Hz ch1-32'),
    11:          (130, '33-64', 'c11: 130Hz ch33-64'),
    GROUP_HI_BR: (400, '33-64', f'c{GROUP_HI_BR}: 400Hz ch33-64'),
}
COND_COLOR = {10: '#f4a3a3', 5: '#a01212', 11: '#9ec4e8', GROUP_HI_BR: '#12407a'}
ORDER = [10, 5, 11, GROUP_HI_BR]
cond_layout = {10: (0, 0), 5: (0, 1), 11: (1, 0), GROUP_HI_BR: (1, 1)}

CONDS_OF_INTEREST = set(ORDER)

# BR22 is a port B control; every condition above is port A. Pooling it mixes two
# stimulating configurations and drags in the tail of the session's firing-rate drift.
CTRL_BR = {2, 4}

# BR12-21: 8 electrodes wide, stepped by 4, spanning ch1-44. Electrode COUNT is constant,
# so injected charge is matched and only position varies -- a pure spatial sweep.
SWEEP = {12: (1, 8),  13: (5, 12),  14: (9, 16),  15: (13, 20), 16: (17, 24),
         17: (21, 28), 18: (25, 32), 19: (29, 36), 20: (33, 40), 21: (37, 44)}
SW = sorted(SWEEP)
CENTER = {br: (lo + hi) / 2 for br, (lo, hi) in SWEEP.items()}

BASE_WIN = (-790, -100)   # stop at -100 to stay clear of the blanking edge


In [ ]:
def load_nprw_rates(npz_path):
    """Load NPRW rates and time axis from a peristim NPZ."""
    npz = np.load(npz_path, allow_pickle=True)
    rates = npz['NPRW_rates_zeroed'].astype(float)  # (trials, 128, T)
    rel_t = npz['NPRW_rel_t'].astype(float)          # (T,)
    br_idx = int(npz['br_idx'])
    stim_dur = 0.0
    try:
        stim_dur = float(np.median(npz['nprw_meta'].item()['stim_dur']))
    except Exception:
        pass
    return rates, rel_t, br_idx, stim_dur


def bcorr(rates, t, win=BASE_WIN):
    """Subtract each trial's own pre-stim mean, per channel."""
    m = (t >= win[0]) & (t <= win[1])
    return rates - np.nanmean(rates[:, :, m], axis=2, keepdims=True)


def align(ta, tb, tol=10.0):
    """Match bins between two time axes by nearest centre. tol = half a bin."""
    ia, ib = [], []
    for i, v in enumerate(ta):
        d = np.abs(tb - v)
        j = int(np.argmin(d))
        if d[j] <= tol:
            ia.append(i); ib.append(j)
    return np.array(ia), np.array(ib)


def segs_of(t):
    """Contiguous index blocks, split at the blanking gap."""
    return np.split(np.arange(len(t)), np.where(np.diff(t) > 25)[0] + 1)


def bin_edges(t):
    e = np.empty(len(t) + 1)
    e[1:-1] = (t[:-1] + t[1:]) / 2
    e[0]  = t[0]  - (t[1] - t[0]) / 2
    e[-1] = t[-1] + (t[-1] - t[-2]) / 2
    return e


def draw_heat(ax, d, t, vmax, cmap='RdBu_r', ny=None):
    ny = d.shape[0] if ny is None else ny
    m = None
    for sg in segs_of(t):
        if len(sg) < 2:
            continue
        m = ax.pcolormesh(bin_edges(t[sg]), np.arange(ny + 1), d[:, sg],
                          cmap=cmap, vmin=-vmax, vmax=vmax, shading='flat', rasterized=True)
    g = np.where(np.diff(t) > 25)[0]
    if len(g):
        ax.axvspan(t[g[0]], t[g[0] + 1], color='0.85', zorder=3)
    ax.axvline(0, color='k', ls='--', lw=0.8, alpha=0.6, zorder=4)
    ax.set_xlim(t[0], t[-1])
    return m


def windows_in(t, width=4, step=1):
    """Sliding windows that never straddle the gap."""
    out = []
    for sg in segs_of(t):
        for i in range(0, len(sg) - width + 1, step):
            out.append(sg[i:i + width])
    return out


In [ ]:
# Load target_B NPZs: the four 2x2 conditions plus the port A controls
stim_data = {}  # br_idx -> (rates, rel_t, stim_dur)
ctrl_data = {}  # br_idx -> (rates, rel_t)

for f in sorted(stim_dir.glob('*.npz')):
    rates, rel_t, br_idx, sd = load_nprw_rates(f)
    if br_idx in CONDS_OF_INTEREST:
        stim_data[br_idx] = (rates, rel_t, sd)
        print(f'BR{br_idx:03d}: {rates.shape}, stim_dur={sd:.0f}ms, t=[{rel_t[0]:.0f}, {rel_t[-1]:.0f}]')

for f in sorted(ctrl_dir.glob('*.npz')):
    rates, rel_t, br_idx, _ = load_nprw_rates(f)
    if br_idx not in CTRL_BR:
        continue
    ctrl_data[br_idx] = (rates, rel_t)
    print(f'ctrl BR{br_idx:03d}: {rates.shape}, t=[{rel_t[0]:.0f}, {rel_t[-1]:.0f}]')

print(f'\nStim conds loaded: {sorted(stim_data.keys())}')
print(f'Ctrl files loaded: {sorted(ctrl_data.keys())}  (BR22 excluded, it is port B)')


In [18]:
# Inspect time axes — stim vs ctrl should have different lengths (gap vs no gap)
for br, (r, t, sd) in stim_data.items():
    dt = np.diff(t)
    gaps = np.where(dt > 25)[0]  # bins >25ms apart = gap
    print(f'cond {br}: {len(t)} bins, stim_dur={sd:.0f}ms', end='')
    if len(gaps):
        g = gaps[0]
        print(f', gap at bin {g}: t[{g}]={t[g]:.0f} -> t[{g+1}]={t[g+1]:.0f} ({dt[g]:.0f}ms)')
    else:
        print(', no gap')

for br, (r, t) in ctrl_data.items():
    dt = np.diff(t)
    gaps = np.where(dt > 25)[0]
    print(f'ctrl {br}: {len(t)} bins', end='')
    if len(gaps):
        g = gaps[0]
        print(f', gap at bin {g}: t[{g}]={t[g]:.0f} -> t[{g+1}]={t[g+1]:.0f}')
    else:
        print(', no gap')

cond 5: 63 bins, stim_dur=99ms, gap at bin 38: t[38]=-30 -> t[39]=129 (159ms)
cond 7: 63 bins, stim_dur=99ms, gap at bin 38: t[38]=-30 -> t[39]=129 (159ms)
cond 10: 63 bins, stim_dur=93ms, gap at bin 38: t[38]=-30 -> t[39]=123 (153ms)
cond 11: 63 bins, stim_dur=93ms, gap at bin 38: t[38]=-30 -> t[39]=123 (153ms)
ctrl 2: 70 bins, no gap
ctrl 4: 70 bins, no gap
ctrl 22: 70 bins, no gap


## Delta FR: stim - ctrl

For each condition, compute mean FR across trials per (channel, time), then subtract control mean.
Control has 70 bins (no gap), stim has 63 bins (with gap) — need to align by `rel_t` values, not by index.

In [ ]:
# Pool control trials (port A only) and baseline-correct each trial against its own pre-stim.
ctrl_list, ctrl_t = [], None
for br, (r, t) in sorted(ctrl_data.items()):
    ctrl_list.append(r)
    if ctrl_t is None:
        ctrl_t = t
    else:
        assert np.allclose(ctrl_t, t), f'ctrl time axes differ: BR{br}'

ctrl_all = np.concatenate(ctrl_list, axis=0)
ctrl_mean = np.nanmean(ctrl_all, axis=0)                      # raw, kept for the diagnostic below
ctrl_mean_bc = np.nanmean(bcorr(ctrl_all, ctrl_t), axis=0)    # what everything downstream uses
print(f'Control: {ctrl_all.shape[0]} trials from BR{sorted(ctrl_data)}, {ctrl_all.shape[2]} bins')

# Held-out check on whether baseline correction really removes the drift offset. Fitting and
# scoring on the same window returns exactly 0 by construction, so fit on the early part of
# pre-stim and score on the late part. Each channel is averaged over the whole scoring window
# first -- the max over single (channel, bin) cells is 20 ms binning noise at these n.
FIT_WIN, TEST_WIN = (-790, -300), (-280, -100)

def maxch(d, t, win):
    m = (t >= win[0]) & (t <= win[1])
    return np.nanmax(np.abs(np.nanmean(d[:, m], axis=1)))

ctrl_fit = np.nanmean(bcorr(ctrl_all, ctrl_t, FIT_WIN), axis=0)
print()
print(f'held-out pre-stim check   fit {FIT_WIN} ms  ->  score {TEST_WIN} ms')
print(f'{"":26s}{"pre-stim FR":>13s}{"raw max|ch|":>13s}{"corrected":>12s}')
for br in ORDER:
    r, t, _ = stim_data[br]
    isx, ic = align(t, ctrl_t)
    ts = t[isx]
    fr0 = np.nanmean(r[:, :, (t >= BASE_WIN[0]) & (t <= BASE_WIN[1])])
    d_raw = np.nanmean(r, axis=0)[:, isx] - ctrl_mean[:, ic]
    d_fit = np.nanmean(bcorr(r, t, FIT_WIN), axis=0)[:, isx] - ctrl_fit[:, ic]
    print(f'{COND_MAP[br][2]:26s}{fr0:13.2f}{maxch(d_raw, ts, TEST_WIN):13.2f}'
          f'{maxch(d_fit, ts, TEST_WIN):12.2f}')


In [ ]:
# delta FR = baseline-corrected stim mean - baseline-corrected control mean.
# Stim post-gap bins sit ~1 ms off the ctrl grid (129 vs 130), hence the nearest-bin match.
delta_fr, shared_t, delta_trials = {}, {}, {}

for br, (rates, t_stim, sd) in stim_data.items():
    isx, ic = align(t_stim, ctrl_t)
    s_bc = bcorr(rates, t_stim)[:, :, isx]
    delta_trials[br] = s_bc - ctrl_mean_bc[None, :, ic]     # (trials, 128, T)
    delta_fr[br] = np.nanmean(delta_trials[br], axis=0)     # (128, T)
    shared_t[br] = t_stim[isx]
    n_pre = int(np.sum(shared_t[br] < -30))
    n_post = int(np.sum(shared_t[br] > 100))
    print(f'{COND_MAP[br][2]:26s} {rates.shape[0]:2d} trials, '
          f'{len(isx)} shared bins ({n_pre} pre + {n_post} post)')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharey=True)
vmax = max(np.nanpercentile(np.abs(d), 95) for d in delta_fr.values())

for br, (row, col) in cond_layout.items():
    ax = axes[row, col]
    m = draw_heat(ax, delta_fr[br], shared_t[br], vmax, ny=128)
    ax.set_title(COND_MAP[br][2], fontsize=10)
    if col == 0:
        ax.set_ylabel('Channel')
    if row == 1:
        ax.set_xlabel('Time (ms)')

fig.suptitle('\u0394FR: stim \u2212 control  (per-trial baseline corrected, port A controls only)\n'
             'NRR_RW011 / target_B / NPRW', fontsize=12)
fig.tight_layout(rect=[0, 0, 0.90, 1])

cax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(m, cax=cax, label='\u0394FR (spk/s)')
plt.show()


## Population traces

Collapsing 128 channels to their mean answers "how much", not "which pattern". Two conditions
with opposite patterns give the same trace. Kept as a sanity view only — the pattern-level
question is handled by the RSA cells further down.


In [ ]:
GROUPS = [('ch 1-32 stimulated',  [10, 5]),
          ('ch 33-64 stimulated', [11, GROUP_HI_BR])]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (gname, brs) in zip(axes, GROUPS):
    for br in brs:
        mu, t = np.nanmean(delta_fr[br], axis=0), shared_t[br]
        n = stim_data[br][0].shape[0]
        for k, sg in enumerate(segs_of(t)):
            ax.plot(t[sg], mu[sg], color=COND_COLOR[br], lw=2,
                    label=f'{COND_MAP[br][0]} Hz  (n={n})' if k == 0 else None)
    ax.axhline(0, color='k', lw=0.6)
    ax.axvline(0, color='k', ls='--', lw=0.8, alpha=0.5)
    ax.axvspan(-30, 129, color='0.9', zorder=0)
    ax.set_xlabel('Time (ms)')
    ax.set_title(gname, fontsize=10)
    ax.legend(fontsize=8)
axes[0].set_ylabel('\u0394FR, mean over 128 ch (spk/s)')

lo = min(a.get_ylim()[0] for a in axes); hi = max(a.get_ylim()[1] for a in axes)
for a in axes:
    a.set_ylim(lo, hi)

fig.suptitle('Population \u0394FR vs control \u2014 one panel per stimulated electrode group\n'
             'darker = 400 Hz, lighter = 130 Hz', fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
from scipy.stats import trim_mean

trace = {br: np.nanmean(delta_trials[br], axis=1) for br in ORDER}   # (trials, T)

fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.55], hspace=0.35, wspace=0.28)
vmax = max(np.nanpercentile(np.abs(v), 97) for v in trace.values())

for br, (row, col) in cond_layout.items():
    ax = fig.add_subplot(gs[row, col])
    n = trace[br].shape[0]
    m = draw_heat(ax, trace[br], shared_t[br], vmax, ny=n)
    ax.set_yticks(np.arange(n) + 0.5)
    ax.set_yticklabels(np.arange(1, n + 1), fontsize=6)
    ax.set_ylabel('trial')
    ax.set_title(COND_MAP[br][2], fontsize=9)
    if row == 1:
        ax.set_xlabel('Time (ms)')

axc = fig.add_subplot(gs[:, 2])
rng = np.random.RandomState(0)
for i, br in enumerate(ORDER):
    v = np.nanmean(trace[br][:, shared_t[br] > 100], axis=1)
    axc.scatter(np.full(len(v), i) + rng.uniform(-.10, .10, len(v)),
                v, s=32, color=COND_COLOR[br], edgecolor='k', linewidth=.4, zorder=3)
    axc.hlines(v.mean(), i - .28, i + .28, color='k', lw=2, zorder=4)
axc.axhline(0, color='k', lw=.6)
axc.set_xticks(range(4))
axc.set_xticklabels([f'c{b}\n{COND_MAP[b][0]}Hz\n{COND_MAP[b][1]}' for b in ORDER], fontsize=7)
axc.set_ylabel('post-stim mean \u0394FR (spk/s)')
axc.set_title('per-trial post-stim (t > 100 ms)', fontsize=9)

fig.suptitle('Per-trial \u0394FR \u2014 is the effect carried by all trials, or one outlier?', fontsize=12)
cax = fig.add_axes([0.355, 0.02, 0.25, 0.014])
fig.colorbar(m, cax=cax, orientation='horizontal', label='\u0394FR (spk/s)')
plt.show()

# Report robust statistics instead of deleting trials -- picking an exclusion rule after
# seeing the figure is circular, and the final statistic (Mann-Whitney AUC) is rank-based.
print(f'{"":26s}{"n":>4s}{"mean":>9s}{"trim20%":>9s}{"median":>9s}{"drop-max":>10s}')
for br in ORDER:
    v = np.nanmean(trace[br][:, shared_t[br] > 100], axis=1)
    print(f'{COND_MAP[br][2]:26s}{len(v):4d}{v.mean():+9.2f}'
          f'{trim_mean(v, 0.2):+9.2f}{np.median(v):+9.2f}{v[v != v.max()].mean():+10.2f}')


In [ ]:
CONTRASTS = [
    (GROUP_HI_BR, 5,  'fixed 400 Hz\nch33-64 \u2212 ch1-32'),
    (11, 10,          'fixed 130 Hz\nch33-64 \u2212 ch1-32'),
    (5,  10,          'fixed ch1-32\n400 Hz \u2212 130 Hz'),
    (GROUP_HI_BR, 11, 'fixed ch33-64\n400 Hz \u2212 130 Hz'),
]

cmean = {br: np.nanmean(bcorr(stim_data[br][0], stim_data[br][1]), axis=0) for br in ORDER}
D, T = {}, {}
for a, b, _ in CONTRASTS:
    ia, ib = align(stim_data[a][1], stim_data[b][1])
    D[(a, b)] = cmean[a][:, ia] - cmean[b][:, ib]
    T[(a, b)] = (stim_data[a][1][ia] + stim_data[b][1][ib]) / 2

fig, axes = plt.subplots(2, 2, figsize=(15, 8.5), sharey=True)
vmax = max(np.nanpercentile(np.abs(v), 97) for v in D.values())
for k, (a, b, lab) in enumerate(CONTRASTS):
    ax = axes[k // 2, k % 2]
    m = draw_heat(ax, D[(a, b)], T[(a, b)], vmax, cmap='PuOr_r', ny=128)
    ax.set_title(lab, fontsize=9.5)
    if k % 2 == 0:
        ax.set_ylabel('Channel')
    if k // 2 == 1:
        ax.set_xlabel('Time (ms)')

fig.suptitle('Effect of changing one factor while the other is held fixed\n'
             'top row = electrode group swapped   |   bottom row = frequency swapped', fontsize=12)
fig.tight_layout(rect=[0, 0, 0.90, 1])
cax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(m, cax=cax, label='\u0394FR difference (spk/s)')
plt.show()


In [ ]:
PANELS = [
    ('Electrode group swapped',
     [(GROUP_HI_BR, 5, '400 Hz held fixed', '#12407a'),
      (11, 10,         '130 Hz held fixed', '#9ec4e8')]),
    ('Frequency swapped',
     [(5, 10,           'ch 1-32 held fixed',  '#a01212'),
      (GROUP_HI_BR, 11, 'ch 33-64 held fixed', '#12407a')]),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (pname, lines) in zip(axes, PANELS):
    for a, b, lab, col in lines:
        mu, t = np.nanmean(D[(a, b)], axis=0), T[(a, b)]
        for k, sg in enumerate(segs_of(t)):
            ax.plot(t[sg], mu[sg], color=col, lw=2, label=lab if k == 0 else None)
    ax.axhline(0, color='k', lw=.6)
    ax.axvline(0, color='k', ls='--', lw=.8, alpha=.5)
    ax.axvspan(-30, 126, color='0.9', zorder=0)
    ax.set_xlabel('Time (ms)')
    ax.set_title(pname, fontsize=10)
    ax.legend(fontsize=8)
axes[0].set_ylabel('\u0394FR difference, mean over 128 ch (spk/s)')

lo = min(a.get_ylim()[0] for a in axes); hi = max(a.get_ylim()[1] for a in axes)
for a in axes:
    a.set_ylim(lo, hi)

fig.suptitle('left: does it matter which electrodes?   right: does it matter which frequency?',
             fontsize=12)
fig.tight_layout()
plt.show()


## Spatial sweep (BR12-21)

Ten blocks, 8 electrodes each, stepped by 4, spanning ch1-44. Frequency, current, duration,
port and behavioural state are all matched; electrode **count** is matched too, so injected
charge is constant and position is the only thing that varies.

This gives the spatial factor 10 levels instead of 2, 105 trials instead of 37, and a model
RDM with graded rather than binary distances — a directional, falsifiable prediction:
representational distance should grow with electrode separation.


In [ ]:
sweep_raw = {}
for f in sorted(stim_dir.glob('*.npz')):
    rates, rel_t, br_idx, _ = load_nprw_rates(f)
    if br_idx in SWEEP:
        sweep_raw[br_idx] = (rates, rel_t)

t_sw = sweep_raw[SW[0]][1]
for br in SW:
    assert np.allclose(sweep_raw[br][1], t_sw), f'BR{br} time axis differs'

print(f'{len(SW)} blocks, {sum(sweep_raw[b][0].shape[0] for b in SW)} trials, '
      f'{len(t_sw)} bins on one shared axis')
for br in SW:
    lo, hi = SWEEP[br]
    print(f'  BR{br}  ch{lo:>2d}-{hi:<2d}  centre {CENTER[br]:>4.1f}  n={sweep_raw[br][0].shape[0]:2d}')


In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize

bc_sw = {br: bcorr(*sweep_raw[br]) for br in SW}
pool = np.concatenate([bc_sw[br] for br in SW], axis=0)
mu_ch = np.nanmean(pool, axis=(0, 2), keepdims=True)
sd_ch = np.nanstd(pool,  axis=(0, 2), keepdims=True)
sd_ch[sd_ch == 0] = 1.0
Z = {br: (bc_sw[br] - mu_ch) / sd_ch for br in SW}

norm = Normalize(min(CENTER.values()), max(CENTER.values()))
fig, ax = plt.subplots(figsize=(10, 5))
for br in SW:
    m = np.nanmean(Z[br], axis=(0, 1))
    for k, sg in enumerate(segs_of(t_sw)):
        ax.plot(t_sw[sg], m[sg], color=cm.viridis(norm(CENTER[br])), lw=1.8,
                label=f'ch{SWEEP[br][0]}-{SWEEP[br][1]}' if k == 0 else None)
ax.axhline(0, color='k', lw=.6)
ax.axvline(0, color='k', ls='--', lw=.8, alpha=.5)
ax.axvspan(-30, 129, color='0.9', zorder=0)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('z-scored rate, mean over 128 ch')
ax.set_title('Sliding 8-electrode window, 400 Hz / 5 \u00b5A / port A\n'
             'colour = position of the stimulated electrodes along the array')
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()


In [ ]:
iu = np.triu_indices(len(SW), k=1)

# two competing accounts of how stimulation position should shape the representation
c  = np.array([CENTER[b] for b in SW])
lo = np.array([SWEEP[b][0] for b in SW])
hi = np.array([SWEEP[b][1] for b in SW])
M_dist = np.abs(c[:, None] - c[None, :])                        # keeps growing with separation
shared = np.clip(np.minimum(hi[:, None], hi[None, :])
                 - np.maximum(lo[:, None], lo[None, :]) + 1, 0, None)
M_ovl = 1 - shared / 8.0                                        # saturates once they stop overlapping

# BR12..BR21 were recorded in that order AND stepped along the array in that order, so
# "electrode separation" and "how far apart in the session" are the same ranking. Anything
# that drifts across the session therefore mimics the distance model exactly.
M_time = np.abs(np.arange(len(SW))[:, None] - np.arange(len(SW))[None, :])
print(f'collinearity of distance and block-order models: '
      f'rho = {stats.spearmanr(M_dist[iu], M_time[iu]).statistic:.3f}')


def partial_spearman(x, y, z):
    """Spearman corr of x and y after regressing the rank of z out of both."""
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    Z = np.column_stack([np.ones_like(rz), rz])
    res = lambda v: v - Z @ np.linalg.lstsq(Z, v, rcond=None)[0]
    return stats.pearsonr(res(rx), res(ry)).statistic


def rdm_at(w):
    X = np.stack([np.nanmean(Z[br][:, :, w], axis=(0, 2)) for br in SW])   # (10, 128)
    return 1 - np.corrcoef(X)


wins = windows_in(t_sw)
tc = np.array([t_sw[w].mean() for w in wins])
rdms = [rdm_at(w) for w in wins]

# the pre-stim geometry is whatever drift produces on its own -- use it as the nuisance model
rdm_pre = np.nanmean([r for r, tt in zip(rdms, tc) if tt < -30], axis=0)

s_dist = np.array([stats.spearmanr(r[iu], M_dist[iu]).statistic for r in rdms])
s_ovl  = np.array([stats.spearmanr(r[iu], M_ovl[iu]).statistic for r in rdms])
s_part = np.array([partial_spearman(r[iu], M_dist[iu], rdm_pre[iu]) for r in rdms])

fig, ax = plt.subplots(figsize=(10, 4.5))
gap = np.where(np.diff(tc) > 25)[0]
series = [(s_dist, '#1b7837', 'electrode distance'),
          (s_ovl,  '#762a83', 'electrode overlap'),
          (s_part, '#d95f02', 'distance, pre-stim geometry partialled out')]
for arr, col, lab in series:
    for k, sg in enumerate(np.split(np.arange(len(tc)), gap + 1)):
        ax.plot(tc[sg], arr[sg], color=col, lw=2, label=lab if k == 0 else None)
ax.axhline(0, color='k', lw=.6)
ax.axvline(0, color='k', ls='--', lw=.8, alpha=.5)
ax.axvspan(-30, 129, color='0.9', zorder=0)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Spearman \u03c1  (neural RDM vs model RDM)')
ax.set_title('Time-resolved RSA \u2014 which account of stimulation position\n'
             'explains the representational geometry, and when')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

pre, post = tc < -30, tc > 129
print(f'\n{"":14s}{"pre-stim":>10s}{"post-stim":>11s}{"gain":>8s}')
for arr, _, lab in series:
    print(f'{lab[:14]:14s}{np.nanmean(arr[pre]):+10.3f}{np.nanmean(arr[post]):+11.3f}'
          f'{np.nanmean(arr[post]) - np.nanmean(arr[pre]):+8.3f}')
print(f'\npeak (partialled) {np.nanmax(s_part[post]):+.3f} @ {tc[post][np.nanargmax(s_part[post])]:.0f} ms')
print('\nPre-stim rho well above zero means drift alone already orders the blocks the way the\n'
      'distance model does. Only the partialled curve, and the pre-to-post gain, are\n'
      'interpretable as a stimulation effect.')


## Next steps

1. Permutation null for the RSA cell — shuffle condition labels at the trial level,
   >=1000 draws, cluster-based correction across the overlapping sliding windows
2. Trial-level RDM (105x105) instead of the 10x10 condition-mean version, so the null has
   something to shuffle and the n-imbalance is visible
3. AUC(channel, time) heatmap — which channels carry the geometry

Open questions for Bryan:

- BR25 and BR27 carry the same label `"130Hz (1-32) Late"` — is BR27 really 33-64?
- Confirm BR7 vs BR9 for the reaching 400 Hz ch33-64 cell (`GROUP_HI_BR` in the condition cell).
  NPZ flags and `cond_label_extras` both say BR7; a session note said BR9.
- BR3 and BR6 are in `skip_conds` and carry no label — what were they?
- BR23/BR24 are the port B replication of the coarse electrode-group contrast, worth running
  once the port A result settles.
